In [ ]:
# Supplementary Figure 2 data process
"""
Functions:
1. Read monthly raw CPS CSV files by year
2. Find the month when each person "first participated in training"
3. Find the most recent occupation (and its month) before training
4. Find the occupation after the first job change following training (and its month)
5. Calculate month differences before/after occupation change
6. Output yearly results and merge into a final dataset

New output columns (compared to original):
- occ_before_month      : Month of the most recent occupation record before training
- occ_after_month       : Month of the first occupation change after training
- months_before_training: Month difference from occ_before_month to training_month
- months_after_training : Month difference from training_month to occ_after_month
- months_occ_to_occ     : Total month difference from occ_before_month to occ_after_month (across training)
"""

import os
import pandas as pd
import numpy as np

# ======================================================
# 0. Basic Path Settings
# ======================================================
base_dir = "Your path/Stranded_Occupations_Replication/Data/raw_new/cps_2010_2023"
yearly_raw_dir = os.path.join(base_dir, "yearly_raw_csv")
yearly_result_dir = os.path.join(base_dir, "yearly_result")
os.makedirs(yearly_result_dir, exist_ok=True)

years = list(range(2010, 2024))

SAVE_YEARLY_MERGED = False
yearly_merged_dir = os.path.join(base_dir, "yearly_merged")
if SAVE_YEARLY_MERGED:
    os.makedirs(yearly_merged_dir, exist_ok=True)

# ======================================================
# 1. Invalid Occupation Code Values
# ======================================================
bad_values = {"", "nan", "none", ".", "-1", "-2", "-3", "-9"}


# ======================================================
# 2. Occupation Code Standardization Function
# ======================================================
def format_occ_code(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    if x.lower() in bad_values:
        return np.nan
    try:
        xf = float(x)
        if xf.is_integer():
            x = str(int(xf))
    except Exception:
        pass
    if x.isdigit():
        return x.zfill(4)
    return x


# ======================================================
# 3. Unify Occupation Variable
# ======================================================
def unify_occ_var(df: pd.DataFrame) -> pd.DataFrame:
    cols_lower = {c.lower(): c for c in df.columns}
    if "ptio1ocd" in cols_lower:
        occ_col = cols_lower["ptio1ocd"]
    elif "peio1ocd" in cols_lower:
        occ_col = cols_lower["peio1ocd"]
    else:
        raise ValueError("Neither ptio1ocd nor peio1ocd found; cannot unify occupation variable.")

    df["occ_code"] = df[occ_col].astype(str).str.strip().str.upper()
    df.loc[df["occ_code"].str.lower().isin(bad_values), "occ_code"] = np.nan
    df["occ_code"] = df["occ_code"].apply(format_occ_code)
    return df


# ======================================================
# 4. Identify Which Columns Actually Exist in a CSV
# ======================================================
def get_existing_usecols(file_path, candidate_cols):
    header = pd.read_csv(file_path, nrows=0)
    header_cols = list(header.columns)
    header_cols_lower = {c.lower(): c for c in header_cols}
    usecols = []
    for col in candidate_cols:
        if col.lower() in header_cols_lower:
            usecols.append(header_cols_lower[col.lower()])
    return usecols


# ======================================================
# Helper: Calculate full months between two datetime columns
#       month_diff(A, B) = Months from A to B (can be negative)
# ======================================================
def month_diff(series_from: pd.Series, series_to: pd.Series) -> pd.Series:
    """
    Return (series_to - series_from) expressed in full months.
    Both columns are datetime64; missing values return pd.NA (Int64).
    """
    diff = (
        (series_to.dt.year - series_from.dt.year) * 12
        + (series_to.dt.month - series_from.dt.month)
    )
    return diff.astype("Int64")   # Integer type that supports NA


# ======================================================
# 5. Process Single Year
# ======================================================
def process_one_year(year: int):
    year_raw_subdir = os.path.join(yearly_raw_dir, str(year))
    year_result_path = os.path.join(
        yearly_result_dir, f"cps_{year}_training_then_occ_switch.csv"
    )

    if not os.path.isdir(year_raw_subdir):
        print(f"[Skip] {year} directory does not exist: {year_raw_subdir}")
        return None

    csv_files = sorted([
        os.path.join(year_raw_subdir, f)
        for f in os.listdir(year_raw_subdir)
        if f.lower().endswith(".csv")
    ])

    if not csv_files:
        print(f"[Skip] No CSV files for {year}")
        return None

    needed_cols = [
        "hryear4", "hrmonth",
        "hrhhid", "hrhhid2", "pulineno",
        "ptio1ocd", "peio1ocd",
        "pelkm1", "pulkm2", "pulkm3", "pulkm4", "pulkm5", "pulkm6"
    ]

    monthly_frames = []

    for f in csv_files:
        usecols = get_existing_usecols(f, needed_cols)
        if len(usecols) == 0:
            print(f"[Skip] {os.path.basename(f)} has no required columns")
            continue

        tmp = pd.read_csv(f, usecols=usecols, low_memory=False)
        tmp.columns = [c.lower() for c in tmp.columns]

        if "hryear4" not in tmp.columns or "hrmonth" not in tmp.columns:
            print(f"[Skip] {os.path.basename(f)} missing hryear4/hrmonth")
            continue

        tmp = tmp[
            (pd.to_numeric(tmp["hryear4"], errors="coerce") == year) &
            (pd.to_numeric(tmp["hrmonth"], errors="coerce").between(1, 12))
        ].copy()

        if tmp.empty:
            continue

        tmp["source_file"] = os.path.basename(f)
        monthly_frames.append(tmp)

    if not monthly_frames:
        print(f"[Skip] No valid monthly data for {year}")
        return None

    df = pd.concat(monthly_frames, ignore_index=True)

    # --------------------------------------------------
    # person_id & year_month
    # --------------------------------------------------
    for col in ["hrhhid", "hrhhid2", "pulineno"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype(str).str.strip()

    df["person_id"] = df["hrhhid"] + "_" + df["hrhhid2"] + "_" + df["pulineno"]
    df["hryear4"] = pd.to_numeric(df["hryear4"], errors="coerce")
    df["hrmonth"] = pd.to_numeric(df["hrmonth"], errors="coerce")
    df["year_month"] = pd.to_datetime(
        dict(year=df["hryear4"], month=df["hrmonth"], day=1),
        errors="coerce"
    )

    df = unify_occ_var(df)

    # --------------------------------------------------
    # training_search (any of pelkm1 / pulkm2-6 == 11)
    # --------------------------------------------------
    search_vars = ["pelkm1", "pulkm2", "pulkm3", "pulkm4", "pulkm5", "pulkm6"]
    existing_search_vars = [v for v in search_vars if v in df.columns]
    if len(existing_search_vars) == 0:
        raise ValueError(f"No PELKM1/PULKM2-6 variables found for {year}.")

    for v in existing_search_vars:
        df[v] = pd.to_numeric(df[v], errors="coerce")

    df["training_search"] = df[existing_search_vars].eq(11).any(axis=1).astype(np.int8)

    df = df.sort_values(["person_id", "year_month"]).reset_index(drop=True)

    # --------------------------------------------------
    # 5.9 First training month per person
    # --------------------------------------------------
    training_first = (
        df.loc[df["training_search"] == 1, ["person_id", "year_month"]]
        .groupby("person_id", as_index=False)
        .agg(training_month=("year_month", "min"))
    )

    if training_first.empty:
        print(f"[Notice] No one with training_search==1 in {year}")
        empty_result = pd.DataFrame(columns=[
            "person_id",
            "training_month", "training_year", "training_month_num",
            "occ_before_training", "occ_before_month",
            "occ_before_year", "occ_before_month_num",
            "occ_after_switch", "occ_after_month",
            "occ_after_year", "occ_after_month_num",
            "switched_after_training",
            "months_before_training", "months_after_training",
            "months_occ_to_occ", "year"
        ])
        empty_result.to_csv(year_result_path, index=False)
        return empty_result

    # Keep only records with valid occupation codes
    df_occ = df.loc[
        df["occ_code"].notna(),
        ["person_id", "year_month", "occ_code"]
    ].copy()

    # --------------------------------------------------
    # 5.11 Most recent occupation (and month) before training (inclusive)
    # --------------------------------------------------
    base_occ = df_occ.merge(training_first, on="person_id", how="inner")
    base_occ = base_occ.loc[
        base_occ["year_month"] <= base_occ["training_month"]
    ].copy()

    base_occ = (
        base_occ.sort_values(["person_id", "year_month"])
        .groupby("person_id", as_index=False)
        .tail(1)      # Last record before training
    )

    # *** New: Keep occ_before_month ***
    base_occ = base_occ[
        ["person_id", "training_month", "year_month", "occ_code"]
    ].rename(columns={
        "year_month": "occ_before_month",
        "occ_code": "occ_before_training"
    })

    # --------------------------------------------------
    # 5.12 Records after training, check for occupation change
    # --------------------------------------------------
    post_train = df_occ.merge(base_occ, on="person_id", how="inner")
    post_train = post_train.loc[
        post_train["year_month"] > post_train["training_month"]
    ].copy()

    post_train["occ_changed_after_training"] = (
        post_train["occ_code"] != post_train["occ_before_training"]
    ).astype(np.int8)

    # --------------------------------------------------
    # 5.13 First occupation change after training (and month)
    # --------------------------------------------------
    switch_after = post_train.loc[
        post_train["occ_changed_after_training"] == 1
    ].copy()

    first_switch_after = (
        switch_after.sort_values(["person_id", "year_month"])
        .groupby("person_id", as_index=False)
        .head(1)
    )

    # *** New: Keep occ_after_month ***
    first_switch_after = first_switch_after[
        ["person_id", "occ_before_training", "year_month", "occ_code"]
    ].rename(columns={
        "year_month": "occ_after_month",
        "occ_code": "occ_after_switch"
    })

    # --------------------------------------------------
    # 5.14 Merge into final result dataset
    # --------------------------------------------------
    result = base_occ.merge(
        first_switch_after,
        on=["person_id", "occ_before_training"],
        how="left"
    )

    result["switched_after_training"] = (
        result["occ_after_switch"].notna().astype(np.int8)
    )

    # --------------------------------------------------
    # Calculate month differences (datetime, correct across years)
    # (1) Occupation month before training → Training month
    result["months_before_training"] = month_diff(
        result["occ_before_month"], result["training_month"]
    )
    # (2) Training month → Occupation change month
    result["months_after_training"] = month_diff(
        result["training_month"], result["occ_after_month"]
    )
    # (3) Occupation month before training → Occupation change month (total span)
    result["months_occ_to_occ"] = month_diff(
        result["occ_before_month"], result["occ_after_month"]
    )

    # --------------------------------------------------
    # Split year/month for readability and Stata/R usage
    result["training_year"]      = result["training_month"].dt.year
    result["training_month_num"] = result["training_month"].dt.month

    result["occ_before_year"]      = result["occ_before_month"].dt.year
    result["occ_before_month_num"] = result["occ_before_month"].dt.month

    result["occ_after_year"]      = result["occ_after_month"].dt.year
    result["occ_after_month_num"] = result["occ_after_month"].dt.month

    result["year"] = year   # Source year (year training occurred)

    result = result[[
        "person_id",
        # --- Training timing ---
        "training_month",           # datetime
        "training_year",            # split
        "training_month_num",       # split
        # --- Occupation before training ---
        "occ_before_training",
        "occ_before_month",         # datetime
        "occ_before_year",          # split
        "occ_before_month_num",     # split
        # --- Occupation after training ---
        "occ_after_switch",
        "occ_after_month",          # datetime
        "occ_after_year",           # split
        "occ_after_month_num",      # split
        # --- Flags and month differences ---
        "switched_after_training",
        "months_before_training",
        "months_after_training",
        "months_occ_to_occ",
        "year"
    ]].copy()

    result.to_csv(year_result_path, index=False)

    if SAVE_YEARLY_MERGED:
        year_merge_path = os.path.join(
            yearly_merged_dir, f"cps_{year}_merged.csv"
        )
        df.to_csv(year_merge_path, index=False)

    print(
        f"[Complete] {year}: n={len(result)}, "
        f"switched={int(result['switched_after_training'].sum())}, "
        f"avg months_after_training="
        f"{result['months_after_training'].mean():.1f}"
    )
    return result


# ======================================================
# 6. Main: Process year by year and merge final dataset
# ======================================================
all_results = []

for year in years:
    try:
        out = process_one_year(year)
        if out is not None and len(out) > 0:
            all_results.append(out)
    except Exception as e:
        print(f"[Processing failed] {year}: {e}")

if len(all_results) > 0:
    final_panel = pd.concat(all_results, ignore_index=True)
else:
    final_panel = pd.DataFrame(columns=[
        "person_id",
        "training_month", "training_year", "training_month_num",
        "occ_before_training", "occ_before_month",
        "occ_before_year", "occ_before_month_num",
        "occ_after_switch", "occ_after_month",
        "occ_after_year", "occ_after_month_num",
        "switched_after_training",
        "months_before_training", "months_after_training",
        "months_occ_to_occ", "year"
    ])
    
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

final_panel_path = os.path.join(
    temp_dir, "cps_2010_2023_training_then_occ_switch_all_years.csv"
)
final_panel.to_csv(final_panel_path, index=False)

print("\n[Final dataset exported]", final_panel_path)
print("[Total rows]", len(final_panel))
print(final_panel.head(20))

# --------------------------------------------------
# Simple descriptive statistics (only for those who changed jobs)
# --------------------------------------------------
switched = final_panel[final_panel["switched_after_training"] == 1].copy()
print("\n[Month difference statistics for those who changed occupations]")
print(switched[
    ["months_before_training", "months_after_training", "months_occ_to_occ"]
].describe())

In [ ]:
# Supplementary Figure 2 data process
"""
Functions:
1. Read monthly raw CPS CSV files by year
2. Find the month when each person “first participated in training”
3. Find the most recent occupation before training
4. Find the occupation after the first job change following training
5. Output yearly results and merge into a final dataset

Speed improvements compared to original code:
- Read only necessary columns instead of the entire table
- Do NOT export yearly merged large files by default to reduce disk I/O
- Filter essential information monthly before concatenation
- Minimize repeated merges, sorts, and intermediate large objects
"""

import os
import pandas as pd
import numpy as np

# ======================================================
# 0. Basic Path Settings
# ======================================================
base_dir = "Your path/Stranded_Occupations_Replication/Data/raw_new/cps_2010_2023"
yearly_raw_dir = os.path.join(base_dir, "yearly_raw_csv")
yearly_result_dir = os.path.join(base_dir, "yearly_result")
os.makedirs(yearly_result_dir, exist_ok=True)

# Process years: 2010-2023
years = list(range(2010, 2024))

# Whether to export merged yearly files
# Recommended to set False first for speed; set True later if you need to check raw merged data
SAVE_YEARLY_MERGED = False
yearly_merged_dir = os.path.join(base_dir, "yearly_merged")
if SAVE_YEARLY_MERGED:
    os.makedirs(yearly_merged_dir, exist_ok=True)

# ======================================================
# 1. Invalid Occupation Code Values
# ======================================================
bad_values = {"", "nan", "none", ".", "-1", "-2", "-3", "-9"}


# ======================================================
# 2. Occupation Code Standardization Function
#    Purpose: Standardize occupation codes into a comparable format
# ======================================================
def format_occ_code(x):
    """
    Standardize occupation code format:
    - Convert missing values to np.nan
    - Strip whitespace and convert to uppercase
    - Pad numeric codes to 4 digits with leading zeros
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    if x.lower() in bad_values:
        return np.nan

    # Some values may be read as float, e.g., 123.0
    try:
        xf = float(x)
        if xf.is_integer():
            x = str(int(xf))
    except Exception:
        pass

    # If purely numeric, pad to 4 digits
    if x.isdigit():
        return x.zfill(4)

    return x


# ======================================================
# 3. Unify Occupation Variable
#    Purpose: Occupation variable may be named ptio1ocd or peio1ocd across years
# ======================================================
def unify_occ_var(df: pd.DataFrame) -> pd.DataFrame:
    """
    Automatically detect occupation variable by column name:
    - Use ptio1ocd first
    - Fall back to peio1ocd
    Finally create a unified 'occ_code' column
    """
    cols_lower = {c.lower(): c for c in df.columns}

    if "ptio1ocd" in cols_lower:
        occ_col = cols_lower["ptio1ocd"]
    elif "peio1ocd" in cols_lower:
        occ_col = cols_lower["peio1ocd"]
    else:
        raise ValueError("Neither ptio1ocd nor peio1ocd found; cannot unify occupation variable.")

    df["occ_code"] = df[occ_col].astype(str).str.strip().str.upper()
    df.loc[df["occ_code"].str.lower().isin(bad_values), "occ_code"] = np.nan
    df["occ_code"] = df["occ_code"].apply(format_occ_code)

    return df


# ======================================================
# 4. Identify Which Columns Actually Exist in a CSV
#    Purpose: Read only necessary columns to improve speed
# ======================================================
def get_existing_usecols(file_path, candidate_cols):
    """
    Read only the header first to check which required columns exist,
    then return the usecols list for read_csv
    """
    header = pd.read_csv(file_path, nrows=0)
    header_cols = list(header.columns)
    header_cols_lower = {c.lower(): c for c in header_cols}

    usecols = []
    for col in candidate_cols:
        if col.lower() in header_cols_lower:
            usecols.append(header_cols_lower[col.lower()])

    return usecols


# ======================================================
# 5. Process Single Year
#    Core logic:
#    (1) Read all months for the year
#    (2) Find first training month per person
#    (3) Find most recent occupation before training
#    (4) Find first occupation change after training
# ======================================================
def process_one_year(year: int):
    year_raw_subdir = os.path.join(yearly_raw_dir, str(year))
    year_result_path = os.path.join(
        yearly_result_dir, f"cps_{year}_training_then_occ_switch.csv"
    )

    if SAVE_YEARLY_MERGED:
        year_merge_path = os.path.join(
            yearly_merged_dir, f"cps_{year}_merged.csv"
        )

    # --------------------------------------------------
    # 5.1 Check if yearly directory exists
    # --------------------------------------------------
    if not os.path.isdir(year_raw_subdir):
        print(f"[Skip] {year} directory does not exist: {year_raw_subdir}")
        return None

    csv_files = sorted([
        os.path.join(year_raw_subdir, f)
        for f in os.listdir(year_raw_subdir)
        if f.lower().endswith(".csv")
    ])

    if not csv_files:
        print(f"[Skip] No CSV files for {year}")
        return None

    # --------------------------------------------------
    # 5.2 Define only columns we actually need
    #     This makes read_csv much faster
    # --------------------------------------------------
    needed_cols = [
        "hryear4", "hrmonth",
        "hrhhid", "hrhhid2", "pulineno",
        "ptio1ocd", "peio1ocd",
        "pelkm1", "pulkm2", "pulkm3", "pulkm4", "pulkm5", "pulkm6"
    ]

    monthly_frames = []

    # --------------------------------------------------
    # 5.3 Read month by month: only read necessary columns to reduce memory and time
    # --------------------------------------------------
    for f in csv_files:
        usecols = get_existing_usecols(f, needed_cols)

        # Skip if no key columns
        if len(usecols) == 0:
            print(f"[Skip] {os.path.basename(f)} has no required columns")
            continue

        tmp = pd.read_csv(
            f,
            usecols=usecols,
            low_memory=False
        )

        # Standardize column names to lowercase for easier processing
        tmp.columns = [c.lower() for c in tmp.columns]

        # Skip if year/month columns are missing
        if "hryear4" not in tmp.columns or "hrmonth" not in tmp.columns:
            print(f"[Skip] {os.path.basename(f)} missing hryear4/hrmonth")
            continue

        # Keep only target year and valid months (1-12)
        tmp = tmp[
            (pd.to_numeric(tmp["hryear4"], errors="coerce") == year) &
            (pd.to_numeric(tmp["hrmonth"], errors="coerce").between(1, 12))
        ].copy()

        if tmp.empty:
            continue

        tmp["source_file"] = os.path.basename(f)
        monthly_frames.append(tmp)

    # Return empty if no valid data for the year
    if not monthly_frames:
        print(f"[Skip] No valid monthly data for {year}")
        return None

    # --------------------------------------------------
    # 5.4 Concatenate full-year data
    # --------------------------------------------------
    df = pd.concat(monthly_frames, ignore_index=True)

    # --------------------------------------------------
    # 5.5 Create person_id and year_month
    #     person_id = household ID + sub-ID + line number
    # --------------------------------------------------
    for col in ["hrhhid", "hrhhid2", "pulineno"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype(str).str.strip()

    df["person_id"] = df["hrhhid"] + "_" + df["hrhhid2"] + "_" + df["pulineno"]

    df["hryear4"] = pd.to_numeric(df["hryear4"], errors="coerce")
    df["hrmonth"] = pd.to_numeric(df["hrmonth"], errors="coerce")

    df["year_month"] = pd.to_datetime(
        dict(year=df["hryear4"], month=df["hrmonth"], day=1),
        errors="coerce"
    )

    # --------------------------------------------------
    # 5.6 Unify occupation variable
    # --------------------------------------------------
    df = unify_occ_var(df)

    # --------------------------------------------------
    # 5.7 Identify training_search
    #     Rule: any of pelkm1 / pulkm2-6 == 11
    # --------------------------------------------------
    search_vars = ["pelkm1", "pulkm2", "pulkm3", "pulkm4", "pulkm5", "pulkm6"]
    existing_search_vars = [v for v in search_vars if v in df.columns]

    if len(existing_search_vars) == 0:
        raise ValueError(f"No PELKM1/PULKM2-6 variables found for {year}.")

    for v in existing_search_vars:
        df[v] = pd.to_numeric(df[v], errors="coerce")

    df["training_search"] = df[existing_search_vars].eq(11).any(axis=1).astype(np.int8)

    # --------------------------------------------------
    # 5.8 Sort
    #     Required for finding "first training" and "most recent occupation before training"
    # --------------------------------------------------
    df = df.sort_values(["person_id", "year_month"]).reset_index(drop=True)

    # --------------------------------------------------
    # 5.9 Find first training month per person
    # --------------------------------------------------
    training_first = (
        df.loc[df["training_search"] == 1, ["person_id", "year_month"]]
        .groupby("person_id", as_index=False)
        .agg(training_month=("year_month", "min"))
    )

    # Output empty result if no one had training this year
    if training_first.empty:
        print(f"[Notice] No one with training_search==1 in {year}")

        empty_result = pd.DataFrame(columns=[
            "person_id", "training_month", "occ_before_training",
            "occ_after_switch", "switched_after_training", "year"
        ])
        empty_result.to_csv(year_result_path, index=False)
        return empty_result

    # --------------------------------------------------
    # 5.10 Keep only records with valid occupation codes
    #      Because occupation comparisons only use valid records
    # --------------------------------------------------
    df_occ = df.loc[df["occ_code"].notna(), ["person_id", "year_month", "occ_code"]].copy()

    # --------------------------------------------------
    # 5.11 Find most recent occupation on or before training month
    #      This is occ_before_training
    # --------------------------------------------------
    base_occ = df_occ.merge(training_first, on="person_id", how="inner")
    base_occ = base_occ.loc[base_occ["year_month"] <= base_occ["training_month"]].copy()

    # Keep the last occupation record before training for each person
    base_occ = (
        base_occ.sort_values(["person_id", "year_month"])
        .groupby("person_id", as_index=False)
        .tail(1)
    )

    base_occ = base_occ[["person_id", "training_month", "occ_code"]].rename(
        columns={"occ_code": "occ_before_training"}
    )

    # --------------------------------------------------
    # 5.12 Find records after training
    #      Then check if occupation changed relative to before training
    # --------------------------------------------------
    post_train = df_occ.merge(base_occ, on="person_id", how="inner")
    post_train = post_train.loc[
        post_train["year_month"] > post_train["training_month"]
    ].copy()

    post_train["occ_changed_after_training"] = (
        post_train["occ_code"] != post_train["occ_before_training"]
    ).astype(np.int8)

    # --------------------------------------------------
    # 5.13 Keep the first record where occupation changed after training
    # --------------------------------------------------
    switch_after = post_train.loc[post_train["occ_changed_after_training"] == 1].copy()

    first_switch_after = (
        switch_after.sort_values(["person_id", "year_month"])
        .groupby("person_id", as_index=False)
        .head(1)
    )

    first_switch_after = first_switch_after[
        ["person_id", "year_month", "occ_before_training", "occ_code"]
    ].rename(columns={
        "year_month": "switch_month_after_training",
        "occ_code": "occ_after_switch"
    })

    # --------------------------------------------------
    # 5.14 Merge into final result dataset
    #      - Those who switched: occ_after_switch is populated
    #      - Those who did not switch: occ_after_switch is null
    # --------------------------------------------------
    result = base_occ.merge(
        first_switch_after,
        on=["person_id", "occ_before_training"],
        how="left"
    )

    result["switched_after_training"] = result["occ_after_switch"].notna().astype(np.int8)
    result["year"] = year

    result = result[[
        "person_id",
        "training_month",
        "occ_before_training",
        "occ_after_switch",
        "switched_after_training",
        "year"
    ]].copy()

    # --------------------------------------------------
    # 5.15 Export results
    #      By default, only export result table, not the large merged file
    # --------------------------------------------------
    result.to_csv(year_result_path, index=False)

    if SAVE_YEARLY_MERGED:
        df.to_csv(year_merge_path, index=False)

    print(f"[Complete] {year}: n={len(result)}, switched={int(result['switched_after_training'].sum())}")
    return result


# ======================================================
# 6. Main: Process year by year and merge final dataset
# ======================================================
all_results = []

for year in years:
    try:
        out = process_one_year(year)
        if out is not None and len(out) > 0:
            all_results.append(out)
    except Exception as e:
        print(f"[Processing failed] {year}: {e}")

# Combine all years
if len(all_results) > 0:
    final_panel = pd.concat(all_results, ignore_index=True)
else:
    final_panel = pd.DataFrame(columns=[
        "person_id", "training_month", "occ_before_training",
        "occ_after_switch", "switched_after_training", "year"
    ])

# Export final dataset
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

final_panel_path = os.path.join(
    temp_dir , "cps_2010_2023_training_then_occ_switch_all_years.csv"
)
final_panel.to_csv(final_panel_path, index=False)

print("\n[Final dataset exported]", final_panel_path)
print("[Total rows]", len(final_panel))
print(final_panel.head(20))

In [ ]:
# Supplementary Figure 2 data process
"""
Functions:
1. Directly read monthly raw CPS CSV files under yearly_raw_csv/2010-2023
2. No downloading or unzipping, and no dependency on yearly_merged files
3. Analyze year by year for 2010-2023
4. Identify occupation change events
5. Determine if there was any training search before this occupation change
6. Output yearly results and merge into a final dataset

Training group: Individual-level tracking
    → Find first training month
    → Find most recent occupation before training
    → Find first occupation change after training
    → Calculate similarity

No-training group: Monthly comparison
    → Identify occupation changes between adjacent months
    → Exclude individuals with any training records
    → Calculate similarity before and after change

Current directory structure:
Your path/cps_2010_2023/
    └── yearly_raw_csv/
          ├── 2010/
          ├── 2011/
          ├── ...
          └── 2023/
"""

import pandas as pd
import numpy as np
import os

# ======================================================
# 1. Path Settings
# ======================================================
# New base_dir: already set to cps_2010_2023
base_dir = "Your path/Stranded_Occupations_Replication/Data/raw_new/cps_2010_2023"

# Raw monthly data directory: pre-downloaded and unzipped
yearly_raw_dir = os.path.join(base_dir, "yearly_raw_csv")

# Output directory: yearly results
yearly_result_dir = os.path.join(base_dir, "yearly_result")
os.makedirs(yearly_result_dir, exist_ok=True)

# Analysis years: 2010-2023
# Note: range(2010, 2024) includes 2023
years = list(range(2010, 2024))

# ======================================================
# 2. Occupation Code Formatting Function
# ======================================================
# Values treated as invalid occupation codes
bad_values = {"", "nan", "none", ".", "-1", "-2", "-3", "-9"}

def format_occ_code(x):
    """
    Purpose:
    Standardize occupation codes for later comparison of occupation changes

    Rules:
    1. Convert missing values to np.nan
    2. Strip whitespace
    3. If numeric, pad to 4 digits, e.g., 123 -> 0123
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x.lower() in bad_values:
        return np.nan

    try:
        xf = float(x)
        if xf.is_integer():
            x = str(int(xf))
    except:
        pass

    if x.isdigit():
        return x.zfill(4)

    return x

# ======================================================
# 3. Load All Monthly Files for One Year and Merge
# ======================================================
def load_one_year_raw_data(year):
    """
    Purpose:
    Go to yearly_raw_csv/{year}/ directory,
    read all monthly CSV files for the year, and concatenate into a yearly DataFrame
    """
    year_dir = os.path.join(yearly_raw_dir, str(year))

    if not os.path.isdir(year_dir):
        print(f"[Skip] {year} directory does not exist: {year_dir}")
        return None

    csv_files = sorted([
        os.path.join(year_dir, f)
        for f in os.listdir(year_dir)
        if f.lower().endswith(".csv")
    ])

    if len(csv_files) == 0:
        print(f"[Skip] No CSV files in {year} directory")
        return None

    year_dfs = []

    for f in csv_files:
        try:
            tmp = pd.read_csv(f, low_memory=False)
            tmp.columns = tmp.columns.str.lower()
            year_dfs.append(tmp)
        except Exception as e:
            print(f"[Warning] Failed to read {os.path.basename(f)}: {e}")

    if len(year_dfs) == 0:
        print(f"[Skip] No monthly files successfully read for {year}")
        return None

    df = pd.concat(year_dfs, ignore_index=True)
    return df

# ======================================================
# 4. Single Year Processing Function
# ======================================================
def process_one_year(year):
    """
    Purpose:
    Process all data for one year and output occupation change results
    """
    outfile = os.path.join(yearly_result_dir, f"cps_{year}_switch_training_final.csv")

    print(f"\n========== Start Processing {year} ==========")

    # --------------------------------------------------
    # 4.1 Read all monthly raw data for this year
    # --------------------------------------------------
    df = load_one_year_raw_data(year)
    if df is None:
        return None

    print("Raw concatenated data shape:", df.shape)

    # --------------------------------------------------
    # 4.2 Keep only Jan-Dec of the current year
    # --------------------------------------------------
    # Convert year/month to numeric to avoid string issues
    df["hryear4"] = pd.to_numeric(df["hryear4"], errors="coerce")
    df["hrmonth"] = pd.to_numeric(df["hrmonth"], errors="coerce")

    df = df[(df["hryear4"] == year) & (df["hrmonth"].between(1, 12))].copy()
    print("Data shape after year filter:", df.shape)

    if df.empty:
        print(f"[Skip] No data left after filtering for {year}")
        return None

    # --------------------------------------------------
    # 4.3 Create person_id
    # --------------------------------------------------
    # person_id identifies the same individual
    for col in ["hrhhid", "hrhhid2", "pulineno"]:
        if col not in df.columns:
            raise ValueError(f"{year} missing key ID variable: {col}")
        df[col] = df[col].astype(str).str.strip()

    df["person_id"] = df["hrhhid"] + "_" + df["hrhhid2"] + "_" + df["pulineno"]

    # --------------------------------------------------
    # 4.4 Create year_month
    # --------------------------------------------------
    # Create proper datetime for stable sorting
    df["year_month"] = pd.to_datetime(
        dict(year=df["hryear4"], month=df["hrmonth"], day=1),
        errors="coerce"
    )

    # --------------------------------------------------
    # 4.5 Unify occupation variable
    # --------------------------------------------------
    # Occupation variable is ptio1ocd in some years, peio1ocd in others
    if "ptio1ocd" in df.columns:
        occ_var = "ptio1ocd"
    elif "peio1ocd" in df.columns:
        occ_var = "peio1ocd"
    else:
        raise ValueError(f"{year} data has neither ptio1ocd nor peio1ocd.")

    print(f"{year} using occupation variable: {occ_var}")

    df[occ_var] = df[occ_var].astype(str).str.strip()
    df.loc[df[occ_var].str.lower().isin(bad_values), occ_var] = np.nan
    df[occ_var] = df[occ_var].apply(format_occ_code)

    # --------------------------------------------------
    # 4.6 Create training_search flag
    # --------------------------------------------------
    # Rule:
    # Flag as training if any of pelkm1 / pulkm2-6 == 11
    search_vars = ["pelkm1", "pulkm2", "pulkm3", "pulkm4", "pulkm5", "pulkm6"]
    existing_search_vars = [v for v in search_vars if v in df.columns]

    if len(existing_search_vars) == 0:
        raise ValueError(f"{year} has no PELKM1/PULKM2-6 variables.")

    for v in existing_search_vars:
        df[v] = pd.to_numeric(df[v], errors="coerce")

    df["training_search"] = df[existing_search_vars].eq(11).any(axis=1).astype(int)

    # --------------------------------------------------
    # 4.7 Keep only employed individuals with valid occupation
    # --------------------------------------------------
    # PREMP = 1/2 typically indicates employed
    if "premp" not in df.columns:
        raise ValueError(f"{year} missing PREMP variable for employment status.")

    df["premp"] = pd.to_numeric(df["premp"], errors="coerce")
    df["employed"] = df["premp"].isin([1, 2]).astype(int)

    df_occ = df[(df["employed"] == 1) & (df[occ_var].notna())].copy()
    df_occ = df_occ.sort_values(["person_id", "year_month"]).copy()

    print("Records available for occupation change analysis:", df_occ.shape[0])

    if df_occ.empty:
        print(f"[Skip] No valid employment records for {year}")
        return None

    # --------------------------------------------------
    # 4.8 Get previous occupation and previous month
    # --------------------------------------------------
    # Use groupby + shift(1) to get prior period occupation for the same person
    df_occ["prev_occ"] = df_occ.groupby("person_id")[occ_var].shift(1)
    df_occ["prev_month"] = df_occ.groupby("person_id")["year_month"].shift(1)

    # --------------------------------------------------
    # 4.9 Determine if occupation changed
    # --------------------------------------------------
    # Mark as change if current occupation != previous occupation
    df_occ["occ_changed"] = np.where(
        df_occ["prev_occ"].notna() & (df_occ[occ_var] != df_occ["prev_occ"]),
        1,
        0
    )

    # --------------------------------------------------
    # 4.10 Determine if any training occurred BEFORE this change
    # --------------------------------------------------
    # Logic:
    # For each person, before current month (exclusive),
    # if any training_search == 1 in history,
    # set ever_training_before_switch = 1
    df_occ["ever_training_before_switch"] = (
        df_occ.groupby("person_id")["training_search"]
        .transform(lambda s: s.shift(1).fillna(0).cummax())
        .astype(int)
    )

    # --------------------------------------------------
    # 4.11 Keep only rows where occupation actually changed
    # --------------------------------------------------
    switch_df = df_occ[df_occ["occ_changed"] == 1].copy()

    # Rename columns for clarity
    switch_df = switch_df.rename(columns={
        "prev_occ": "occ_before",
        occ_var: "occ_after",
        "year_month": "switch_month",
        "prev_month": "before_month"
    })

    # --------------------------------------------------
    # 4.12 Keep final output variables
    # --------------------------------------------------
    final_cols = [
        "person_id",
        "occ_before",
        "before_month",
        "switch_month",
        "occ_after",
        "ever_training_before_switch"
    ]

    final_df = switch_df[final_cols].copy()
    final_df["year"] = year

    # --------------------------------------------------
    # 4.13 Export yearly results
    # --------------------------------------------------
    final_df.to_csv(outfile, index=False)

    print("Exported:", outfile)
    print("Number of occupation change events:", final_df.shape[0])

    return final_df

# ======================================================
# 5. Batch Process 2010-2023
# ======================================================
all_results = []

for year in years:
    try:
        one_year_result = process_one_year(year)
        if one_year_result is not None:
            all_results.append(one_year_result)
    except Exception as e:
        print(f"[Failed] {year}: {e}")

# ======================================================
# 6. Merge Final Dataset
# ======================================================
if len(all_results) > 0:
    all_df = pd.concat(all_results, ignore_index=True)
else:
    all_df = pd.DataFrame(columns=[
        "person_id", "occ_before", "before_month", "switch_month",
        "occ_after", "ever_training_before_switch", "year"
    ])

# Final output file 2010_2023
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

all_outfile = os.path.join(temp_dir, "cps_2010_2023_switch_training_final_all.csv")
all_df.to_csv(all_outfile, index=False)

print("\n==============================")
print("All years processed")
print("Final dataset exported:", all_outfile)
print("Total occupation change events:", all_df.shape[0])
print(all_df.head(20))

In [ ]:
# Supplementary Figure 2 data process
import pandas as pd
import numpy as np
import os
import re

# ======================================================
# 1. File Paths
# ======================================================
switch_file = "Your path/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_training_then_occ_switch_all_years.csv"

crosswalk_file = "Your path/Stranded_Occupations_Replication/Data/raw_new/cps_2010_2023/CPS_2023_Variables.xlsx"
crosswalk_sheet = "职业代码 Occupation Codes"

jskill_file = "Your path/Stranded_Occupations_Replication/Data/temp/jskill_pa.csv"

out_file = "Your path/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_training_then_occ_with_soc_skill.csv"

# ======================================================
# 2. Helper Functions
# ======================================================
def normalize_colnames(df):
    """
    Standardize column names: strip whitespace, convert to lowercase
    For compatibility with 2023 data that may have uppercase column names
    """
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def normalize_census_code(x):
    """
    Standardize census occupation code
    Example: 123 -> 0123
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "."}:
        return np.nan
    try:
        xf = float(x)
        if xf.is_integer():
            x = str(int(xf))
    except Exception:
        pass
    if x.isdigit():
        return x.zfill(4)
    return x

def normalize_soc(x):
    """
    Standardize SOC:
    35-2021 -> 352021
    51-20XX -> 5120XX
    43-4YYY -> 434YYY
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    x = x.replace("-", "").replace(" ", "")
    if x == "" or x.lower() in {"nan", "none", "."}:
        return np.nan
    return x

def add_dash_to_soc(x):
    """
    352021 -> 35-2021
    """
    if pd.isna(x):
        return np.nan
    x = normalize_soc(x)
    if len(x) != 6:
        return x
    return x[:2] + "-" + x[2:]

def has_wildcard(s):
    """
    Treat both X and Y as wildcard characters
    """
    s = normalize_soc(s)
    if pd.isna(s):
        return False
    return ("X" in s) or ("Y" in s)

def soc_pattern_to_regex(s):
    """
    5120XX -> ^5120\d\d$
    434YYY -> ^434\d\d\d$
    """
    s = normalize_soc(s)
    if pd.isna(s):
        return None
    regex = "^"
    for ch in s:
        if ch in {"X", "Y"}:
            regex += r"\d"
        else:
            regex += re.escape(ch)
    regex += "$"
    return regex

def match_soc_codes(all_codes, code_or_pattern):
    """
    Given an exact SOC or pattern with X/Y, return all matching 6-digit SOC codes (no hyphen)
    """
    code_or_pattern = normalize_soc(code_or_pattern)
    if pd.isna(code_or_pattern):
        return []

    if not has_wildcard(code_or_pattern):
        return [code_or_pattern] if code_or_pattern in all_codes else []

    pattern = re.compile(soc_pattern_to_regex(code_or_pattern))
    return [c for c in all_codes if pattern.match(c)]

def convert_trailing_zero_to_x(s):
    """
    If 6-digit SOC ends with 0, treat as X:
    252010 -> 25201X
    412010 -> 41201X
    """
    s = normalize_soc(s)
    if pd.isna(s):
        return np.nan
    if len(s) == 6 and not has_wildcard(s) and s[-1] == "0":
        return s[:5] + "X"
    return s

def find_col_by_keywords(columns, keyword_groups):
    """
    Auto-detect column names
    keyword_groups example:
    [["2020", "census", "code"], ["census", "code"]]
    """
    cols_lower = {c: str(c).strip().lower() for c in columns}
    for col, low in cols_lower.items():
        for kws in keyword_groups:
            if all(k.lower() in low for k in kws):
                return col
    return None

# ======================================================
# 3. Read main dataset (using read_csv)
# ======================================================
df = pd.read_csv(switch_file, low_memory=False)
df = normalize_colnames(df)

# Check required columns
required_cols = ["switched_after_training", "occ_before_training", "occ_after_switch"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Main dataset missing required columns: {missing}")

# Keep only individuals who switched jobs after training
df["switched_after_training"] = pd.to_numeric(df["switched_after_training"], errors="coerce")
df = df[df["switched_after_training"] == 1].copy()

# Standardize before / after census occupation codes
df["occ_before_training"] = df["occ_before_training"].apply(normalize_census_code)
df["occ_after_switch"] = df["occ_after_switch"].apply(normalize_census_code)

print("Sample size after filtering switched_after_training == 1:", len(df))

# ======================================================
# 4. Read crosswalk file
# ======================================================
cw = pd.read_excel(crosswalk_file, sheet_name=crosswalk_sheet)
cw = normalize_colnames(cw)

census_col = find_col_by_keywords(
    cw.columns,
    [
        ["2020", "census", "code"],
        ["census", "code"],
        ["occupation", "code"]
    ]
)

soc_col = find_col_by_keywords(
    cw.columns,
    [
        ["2018", "soc", "code"],
        ["soc", "code"],
        ["2018", "soc"]
    ]
)

if census_col is None or soc_col is None:
    print("Current crosswalk column names:")
    print(list(cw.columns))
    raise ValueError("Cannot auto-detect Census code and 2018 SOC code columns in crosswalk. Please specify manually.")

print("Detected crosswalk columns:")
print("Census code column =", census_col)
print("SOC code column =", soc_col)

cw = cw[[census_col, soc_col]].copy()
cw.columns = ["census_code", "soc_code"]

cw["census_code"] = cw["census_code"].apply(normalize_census_code)
cw["soc_code"] = cw["soc_code"].apply(normalize_soc)

cw = cw.dropna(subset=["census_code", "soc_code"]).drop_duplicates()

# Do not merge directly; create a mapping: one census_code to a list of SOCs
cw_map = (
    cw.groupby("census_code")["soc_code"]
    .apply(lambda s: sorted(set(s.dropna())))
    .to_dict()
)

# Map SOC lists to before/after occupations in main dataset
df["soc_before_list"] = df["occ_before_training"].map(cw_map)
df["soc_after_list"] = df["occ_after_switch"].map(cw_map)

# Process trailing zero -> X for matching
df["soc_before_list_for_match"] = df["soc_before_list"].apply(
    lambda lst: [convert_trailing_zero_to_x(x) for x in lst] if isinstance(lst, list) else np.nan
)
df["soc_after_list_for_match"] = df["soc_after_list"].apply(
    lambda lst: [convert_trailing_zero_to_x(x) for x in lst] if isinstance(lst, list) else np.nan
)

# For preview convenience: get first SOC in list
df["soc_before_first"] = df["soc_before_list"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else np.nan)
df["soc_after_first"] = df["soc_after_list"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else np.nan)

df["soc_before_first_dash"] = df["soc_before_first"].apply(add_dash_to_soc)
df["soc_after_first_dash"] = df["soc_after_first"].apply(add_dash_to_soc)

# ======================================================
# 5. Read skill similarity matrix
# ======================================================
jskill = pd.read_csv(jskill_file, low_memory=False)
jskill = normalize_colnames(jskill)

for col in ["occupation_code", "occupation_code_match"]:
    if col not in jskill.columns:
        raise ValueError(f"jskill_pa.csv missing column: {col}")

sim_candidates = [c for c in jskill.columns if c not in ["occupation_code", "occupation_code_match"]]
if "skill_similarity" in jskill.columns:
    sim_col = "skill_similarity"
elif len(sim_candidates) == 1:
    sim_col = sim_candidates[0]
else:
    print("Current jskill_pa.csv column names:")
    print(list(jskill.columns))
    raise ValueError("Cannot auto-detect skill similarity column. Please specify manually.")

jskill = jskill.rename(columns={sim_col: "skill_similarity"}).copy()

jskill["occupation_code"] = jskill["occupation_code"].apply(normalize_soc)
jskill["occupation_code_match"] = jskill["occupation_code_match"].apply(normalize_soc)
jskill["skill_similarity"] = pd.to_numeric(jskill["skill_similarity"], errors="coerce")

jskill = jskill.dropna(subset=["occupation_code", "occupation_code_match", "skill_similarity"]).copy()

# Add reverse direction for bidirectional matching
jskill_rev = jskill.rename(columns={
    "occupation_code": "occupation_code_match",
    "occupation_code_match": "occupation_code"
})
jskill2 = pd.concat([jskill, jskill_rev], ignore_index=True).drop_duplicates()

all_soc_codes = set(jskill2["occupation_code"]).union(set(jskill2["occupation_code_match"]))

# ======================================================
# 6. Calculate skill similarity
# ======================================================
def expand_soc_candidates(soc_list):
    """
    Given a list of SOCs, expand to all usable 6-digit SOCs for matching
    Supports X / Y wildcards
    """
    if not isinstance(soc_list, list) or len(soc_list) == 0:
        return []

    out = []
    for s in soc_list:
        s = convert_trailing_zero_to_x(s)
        out.extend(match_soc_codes(all_soc_codes, s))

    return sorted(set(out))

def compute_similarity_from_lists(before_soc_list, after_soc_list):
    """
    For one occupation transition record:
    - Before occupation may map to multiple SOCs
    - After occupation may map to multiple SOCs
    Find all matches in jskill and return average similarity
    """
    before_list = expand_soc_candidates(before_soc_list)
    after_list = expand_soc_candidates(after_soc_list)

    if len(before_list) == 0 or len(after_list) == 0:
        return np.nan

    sub = jskill2[
        jskill2["occupation_code"].isin(before_list) &
        jskill2["occupation_code_match"].isin(after_list)
    ].copy()

    if sub.empty:
        return np.nan

    return sub["skill_similarity"].mean()

df["skill_similarity"] = df.apply(
    lambda row: compute_similarity_from_lists(
        row["soc_before_list_for_match"],
        row["soc_after_list_for_match"]
    ),
    axis=1
)

# ======================================================
# 7. Export final result
# ======================================================
df.to_csv(out_file, index=False)

print("Processing complete, output file:")
print(out_file)

preview_cols = [
    "occ_before_training",
    "occ_after_switch",
    "soc_before_first",
    "soc_after_first",
    "soc_before_first_dash",
    "soc_after_first_dash",
    "skill_similarity"
]
preview_cols = [c for c in preview_cols if c in df.columns]

print("\nResult preview:")
print(df[preview_cols].head(20))

In [ ]:
# Supplementary Figure 2A data process
import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm

# ======================================================
# 1. File Paths
# ======================================================
switch_file   = "Your path/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_switch_training_final_all.csv"
crosswalk_file = "Your path/Stranded_Occupations_Replication/Data/raw_new/cps_2010_2023/CPS_2023_Variables.xlsx"
crosswalk_sheet = "职业代码 Occupation Codes"
jskill_file   = "Your path/Stranded_Occupations_Replication/Data/temp/jskill_pa.csv"
out_file      = "Your path/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_notrain_occ_with_soc.csv"


# ======================================================
# 2. Helper Functions
# ======================================================
def normalize_census_code(x):
    if pd.isna(x): return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "."}: return np.nan
    try:
        xf = float(x)
        if xf.is_integer(): x = str(int(xf))
    except Exception: pass
    return x.zfill(4) if x.isdigit() else x

def normalize_soc(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().upper().replace("-", "").replace(" ", "")
    return np.nan if x in {"", "NAN", "NONE", "."} else x

def add_dash_to_soc(x):
    if pd.isna(x): return np.nan
    x = normalize_soc(x)
    return (x[:2] + "-" + x[2:]) if len(x) == 6 else x

def has_wildcard(s):
    return bool(s) and ("X" in s or "Y" in s)

def convert_trailing_zero_to_x(s):
    s = normalize_soc(s)
    if pd.isna(s): return np.nan
    return (s[:5] + "X") if (len(s) == 6 and not has_wildcard(s) and s[-1] == "0") else s

def find_col_by_keywords(columns, keyword_groups):
    for col in columns:
        low = str(col).strip().lower()
        for kws in keyword_groups:
            if all(k.lower() in low for k in kws):
                return col
    return None

# ======================================================
# 3. Read Main Dataset
# ======================================================
print("📂 Reading main dataset...")
df = pd.read_csv(switch_file, low_memory=False)
df.columns = df.columns.str.strip()
print(f"  Original sample size: {len(df):,}")

if "ever_training_before_switch" not in df.columns:
    raise ValueError("Missing column: ever_training_before_switch")

# Keep only individuals with NO training before job switch
df = df[df["ever_training_before_switch"] == 0].copy()
print(f"  After filtering non-training group: {len(df):,} records")

df["occ_before"] = df["occ_before"].apply(normalize_census_code)
df["occ_after"]  = df["occ_after"].apply(normalize_census_code)

# ======================================================
# 4. Read Census -> SOC Crosswalk
# ======================================================
print("\n📂 Reading Census-SOC crosswalk...")
cw = pd.read_excel(crosswalk_file, sheet_name=crosswalk_sheet)

census_col = find_col_by_keywords(cw.columns, [["2020","census","code"],["census","code"],["occupation","code"]])
soc_col    = find_col_by_keywords(cw.columns, [["2018","soc","code"],["soc","code"],["2018","soc"]])

if census_col is None or soc_col is None:
    print("Crosswalk columns:", list(cw.columns))
    raise ValueError("Cannot auto-detect crosswalk columns")

cw = cw[[census_col, soc_col]].copy()
cw.columns = ["census_code", "soc_code"]
cw["census_code"] = cw["census_code"].apply(normalize_census_code)
cw["soc_code"]    = cw["soc_code"].apply(normalize_soc)
cw = cw.dropna().drop_duplicates()
print(f"  Crosswalk entries: {len(cw):,}")

# ======================================================
# 5. Merge SOC Codes (Vectorized, No Loops)
# ======================================================
print("\n🔗 Merging SOC codes...")
df = df.merge(cw.rename(columns={"census_code":"occ_before","soc_code":"soc_before"}), on="occ_before", how="left")
df = df.merge(cw.rename(columns={"census_code":"occ_after", "soc_code":"soc_after"}),  on="occ_after",  how="left")

for col, new in [("soc_before","soc_before_for_match"), ("soc_after","soc_after_for_match")]:
    df[new] = df[col].apply(convert_trailing_zero_to_x)

df["soc_before_dash"]          = df["soc_before"].apply(add_dash_to_soc)
df["soc_after_dash"]           = df["soc_after"].apply(add_dash_to_soc)
df["soc_before_for_match_dash"]= df["soc_before_for_match"].apply(add_dash_to_soc)
df["soc_after_for_match_dash"] = df["soc_after_for_match"].apply(add_dash_to_soc)

print(f"  soc_before match rate: {df['soc_before'].notna().mean():.1%}")
print(f"  soc_after  match rate: {df['soc_after'].notna().mean():.1%}")

# ======================================================
# 6. Read Skill Similarity Matrix
# ======================================================
print("\n📂 Reading skill similarity matrix...")
jskill = pd.read_csv(jskill_file, low_memory=False)

if "skill_similarity" in jskill.columns:
    sim_col = "skill_similarity"
else:
    candidates = [c for c in jskill.columns if c not in ["occupation_code","occupation_code_match"]]
    sim_col = candidates[0] if len(candidates) == 1 else None
    if sim_col is None:
        raise ValueError(f"Cannot identify similarity column, current columns: {list(jskill.columns)}")

jskill = jskill[["occupation_code","occupation_code_match",sim_col]].copy()
jskill.columns = ["occ_a","occ_b","skill_similarity"]
jskill["occ_a"] = jskill["occ_a"].apply(normalize_soc)
jskill["occ_b"] = jskill["occ_b"].apply(normalize_soc)
jskill["skill_similarity"] = pd.to_numeric(jskill["skill_similarity"], errors="coerce")

# Add reverse direction for bidirectional matching
jskill_rev = jskill.rename(columns={"occ_a":"occ_b","occ_b":"occ_a"})
jskill2 = pd.concat([jskill, jskill_rev], ignore_index=True).drop_duplicates(subset=["occ_a","occ_b"])
print(f"  Similarity matrix entries (with reverse): {len(jskill2):,}")

all_soc_codes = set(jskill2["occ_a"]) | set(jskill2["occ_b"])

# ======================================================
# 7. Expand Wildcard SOC & Calculate Similarity
#    Key Optimization: compute unique pairs first, then merge back
# ======================================================
print("\n⚡ Vectorized skill similarity calculation...")

def expand_soc(code):
    """Expand wildcard SOC to list of exact matching codes"""
    code = convert_trailing_zero_to_x(code)
    if pd.isna(code): return []
    if not has_wildcard(code):
        return [code] if code in all_soc_codes else []
    pattern = re.compile("^" + "".join(r"\d" if c in "XY" else re.escape(c) for c in code) + "$")
    return [c for c in all_soc_codes if pattern.match(c)]

# Get unique occupation pairs to avoid redundant calculations
unique_pairs = df[["soc_before_for_match","soc_after_for_match"]].drop_duplicates().dropna()
print(f"  Unique (occ_before, occ_after) pairs: {len(unique_pairs):,} (main table: {len(df):,} records)")

# Pre-expand all unique SOC codes once
all_unique_soc = set(unique_pairs["soc_before_for_match"]) | set(unique_pairs["soc_after_for_match"])
print(f"  Pre-expanding {len(all_unique_soc):,} unique SOC codes...")
expand_cache = {s: expand_soc(s) for s in tqdm(all_unique_soc, desc="  Expanding SOC")}

# Calculate similarity for unique pairs
def compute_pair_similarity(row):
    before_list = expand_cache.get(row["soc_before_for_match"], [])
    after_list  = expand_cache.get(row["soc_after_for_match"], [])
    if not before_list or not after_list:
        return np.nan
    sub = jskill2[jskill2["occ_a"].isin(before_list) & jskill2["occ_b"].isin(after_list)]
    return sub["skill_similarity"].mean() if not sub.empty else np.nan

tqdm.pandas(desc="  Calculating pair similarity")
unique_pairs["skill_similarity"] = unique_pairs.progress_apply(compute_pair_similarity, axis=1)

print(f"  Similarity coverage: {unique_pairs['skill_similarity'].notna().mean():.1%}")

# Merge results back to main dataset
df = df.merge(unique_pairs, on=["soc_before_for_match","soc_after_for_match"], how="left")

# ======================================================
# 8. Export Final Results
# ======================================================
print(f"\n💾 Writing results to {out_file} ...")
df.to_csv(out_file, index=False)
print(f"✅ Complete! Total records: {len(df):,}")

preview_cols = ["occ_before","occ_after","soc_before","soc_after",
                "soc_before_for_match","soc_after_for_match","skill_similarity"]
preview_cols = [c for c in preview_cols if c in df.columns]
print("\nResult preview:")
print(df[preview_cols].head(20).to_string(index=False))
print(f"\nskill_similarity descriptive statistics:")
print(df["skill_similarity"].describe())

In [ ]:
"""
Functions:
1. Read CSV file
2. Extract skill_similarity
3. Plot density distribution
4. Draw vertical line at 90th percentile
5. Label the 90th percentile value
6. Export SVG image
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# 1. File Paths
# =========================
file_path = Path("Your path/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_notrain_occ_with_soc.csv")
out_svg = Path("Your path/Stranded_Occupations_Replication/Data/temp/density_notrain.svg")

# =========================
# 2. Read Data
# =========================
df = pd.read_csv(file_path)

# =========================
# 3. Extract and clean skill_similarity
# =========================
x = pd.to_numeric(df["skill_similarity"], errors="coerce").dropna()

# =========================
# 4. Calculate 90th percentile
# =========================
q90 = x.quantile(0.9)

# =========================
# 5. Plot density distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 5))

# Pandas built-in density plot
x.plot(kind="density", ax=ax)

# =========================
# 6. Draw vertical line at 90th percentile
# =========================
ax.axvline(q90, linestyle="--", linewidth=1, color="dimgray")

# =========================
# 7. Add label
# =========================
ymax = ax.get_ylim()[1]
ax.text(
    q90 + 0.01,
    ymax * 0.9,
    f"90th percentile = {q90:.2f}",
    fontsize=11,
    va="top"
)

# =========================
# 8. Title and axis labels
# =========================
ax.set_xlabel("Skill similarity", fontsize=18)
ax.set_ylabel("Density", fontsize=18)

# Tick label size
ax.tick_params(axis="x", labelsize=15)
ax.tick_params(axis="y", labelsize=15)

# =========================
# 9. Save figure
# =========================
fig.tight_layout()
fig.savefig(out_svg, format="svg", bbox_inches="tight")
plt.show()

# =========================
# 10. Print results
# =========================
print(f"Sample size (non-missing): {len(x)}")
print(f"90th percentile: {q90:.2f}")
print(f"Figure saved to: {out_svg}")

In [ ]:
# Supplenmentary Figure 2B picture drawing
"""
功能：
1. 读取 CSV 文件
2. 提取 skill_similarity
3. 绘制 density 分布图
4. 在 90% 分位数位置画竖线
5. 用标签标注 90% 分位数数值
6. 导出 SVG 图片
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# 1. 文件路径
# =========================
file_path = Path("/Users/huangqing/Documents/paper/strand_labor/Stranded_Occupations_Replication/Data/temp/cps_2010_2023_notrain_occ_with_soc.csv")
out_svg = Path("/Users/huangqing/Documents/paper/strand_labor/Stranded_Occupations_Replication/Data/temp/density_notrain.svg")

# =========================
# 2. 读取数据
# =========================
df = pd.read_csv(file_path)

# =========================
# 3. 提取并清洗 skill_similarity
# =========================
x = pd.to_numeric(df["skill_similarity"], errors="coerce").dropna()

# =========================
# 4. 计算 90% 分位数
# =========================
q90 = x.quantile(0.9)

# =========================
# 5. 绘制 density 分布图
# =========================
fig, ax = plt.subplots(figsize=(8, 5))

# pandas 自带密度图
x.plot(kind="density", ax=ax)

# =========================
# 6. 在 90% 分位数位置画竖线
# =========================
ax.axvline(q90, linestyle="--", linewidth=1, color="dimgray")

# =========================
# 7. 添加标签
# =========================
ymax = ax.get_ylim()[1]
ax.text(
    q90 + 0.01,
    ymax * 0.9,
    f"90th percentile = {q90:.2f}",
    fontsize=11,
    va="top"
)

# =========================
# 8. 图形标题和坐标轴
# =========================
ax.set_xlabel("Skill similarity", fontsize=18)
ax.set_ylabel("Density", fontsize=18)

# x轴数字大小
ax.tick_params(axis="x", labelsize=15)
ax.tick_params(axis="y", labelsize=15)

# =========================
# 9. 保存图片
# =========================
fig.tight_layout()
fig.savefig(out_svg, format="svg", bbox_inches="tight")
plt.show()

# =========================
# 10. 输出结果
# =========================
print(f"样本量（非缺失）: {len(x)}")
print(f"90% 分位数: {q90:.2f}")
print(f"图片已保存: {out_svg}")